# Data Transformation

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, RobustScaler,LabelEncoder
import joblib

In [15]:
train_df = pd.read_csv("../../data/split/train.csv")
val_df = pd.read_csv("../../data/split/validation.csv")
test_df = pd.read_csv("../../data/split/test.csv")
x_train = train_df.drop("diabetes", axis=1)
y_train = train_df["diabetes"]

x_val = val_df.drop("diabetes", axis=1)
y_val = val_df["diabetes"]

x_test = test_df.drop("diabetes", axis=1)
y_test = test_df["diabetes"]

## Feature Discretization

---

### HbA1c Level → `hba1c_category`

Measures average blood sugar over the past 2–3 months.

| Category    | Range         | Clinical Meaning                        |
|-------------|---------------|-----------------------------------------|
| Normal      | < 5.7%        | Healthy glucose metabolism              |
| Prediabetes | 5.7% – 6.4%   | Elevated risk, not yet diabetic         |
| Diabetes    | ≥ 6.5%        | Meets diagnostic threshold for diabetes |

---

### BMI → `bmi_category`

Body Mass Index — ratio of weight to height squared (kg/m²).

| Category       | Range         | Clinical Meaning                              |
|----------------|---------------|-----------------------------------------------|
| Underweight    | < 18.5        | May indicate malnutrition or metabolic issues |
| Healthy Weight | 18.5 – 24.9   | Optimal weight range                          |
| Overweight     | 25.0 – 29.9   | Increased risk of metabolic disorders         |
| Obesity        | ≥ 30.0        | High risk of diabetes, hypertension, CVD      |

---

### Blood Glucose Level → `blood_glucose_category`

Fasting blood glucose measured in mg/dL.

| Category          | Range          | Clinical Meaning                            |
|-------------------|----------------|---------------------------------------------|
| Normal            | 70 – 139 mg/dL | Healthy fasting glucose                     |
| Prediabetes       | 140 – 199 mg/dL| Impaired glucose tolerance                  |
| Possible Diabetes | ≥ 200 mg/dL    | Meets diagnostic threshold, requires workup |

---

### Notes

- `right=False` ensures boundary values fall into the **higher-risk category**
  - e.g. `HbA1c = 6.5` → **Diabetes**, not Prediabetes
  - e.g. `BMI = 30.0` → **Obesity**, not Overweight
- All labels follow **ordinal risk ordering** — suitable for mapping to integers if needed

In [16]:
def discretize(df):
    df["hba1c_category"] = pd.cut(
            df['HbA1c_level'],
            bins=[0, 5.7, 6.5, float("inf")],
            labels=["Normal", "Prediabetes", "Diabetes"],
            right= False
        )
    
    df['bmi_category'] = pd.cut(
            df['bmi'],
            bins=[0, 18.5, 25, 30, float("inf")],
            labels=["Underweight", "Healthy Weight", "Overweight", "Obesity"],
            right= False
        )
    
    df['blood_glucose_category'] = pd.cut(
            df['blood_glucose_level'],
            bins=[70, 140, 200, float("inf")],
            labels=["Normal", "Prediabetes", "Possible Diabetes"],
            right= False
        )
    
    return df

## Feature Encoding 
### Gender Binary Encoding 
- Converts categorical gender to numeric binary
- Female = 0 | Male = 1

In [17]:
x_train["gender"] = x_train["gender"].replace({"Female": 0, "Male": 1})
x_val["gender"] = x_val["gender"].replace({"Female": 0, "Male": 1})
x_test["gender"] = x_test["gender"].replace({"Female": 0, "Male": 1})

### Ordinal Encoding for Discretized Categories
- Maps clinically ordered string categories to integers
- Preserves natural risk ordering: lowest risk = 0 → highest risk = n

In [18]:
def ordinal_encode_categories(df):

    hba1c_mapping = {
        "Normal"      : 0,
        "Prediabetes" : 1,
        "Diabetes"    : 2
    }

    bmi_mapping = {
        "Underweight"    : 0,
        "Healthy Weight" : 1,
        "Overweight"     : 2,
        "Obesity"        : 3
    }

    blood_glucose_mapping = {
        "Normal"            : 0,
        "Prediabetes"       : 1,
        "Possible Diabetes" : 2
    }

    df["hba1c_category"]         = df["hba1c_category"].map(hba1c_mapping).astype(int)
    df["bmi_category"]           = df["bmi_category"].map(bmi_mapping).astype(int)
    df["blood_glucose_category"] = df["blood_glucose_category"].map(blood_glucose_mapping).astype(int)

    return df

## Feature Engineering

### Arithmetic Interaction Features

Captures the **combined effect** of two clinical variables that a model may miss
when evaluating each feature in isolation. Computed as a simple multiplication
of the two source features.

| Feature | Formula | Clinical Rationale |
|---|---|---|
| `glucose_hba1c_interaction` | `blood_glucose_level × HbA1c_level` | Combines short-term and long-term glucose markers — their joint elevation is a stronger diabetes signal than either alone |
| `age_hba1c_interaction` | `age × HbA1c_level` | Captures how HbA1c-related risk compounds with age, reflecting age-dependent metabolic decline |
| `age_bmi_interaction` | `age × bmi` | Reflects how excess body weight becomes increasingly harmful to glucose metabolism at older ages |
| `bmi_hba1c_interaction` | `bmi × HbA1c_level` | Encodes the joint burden of obesity and poor long-term glucose control, a key metabolic syndrome signal |
| `age_glucose_interaction` | `age × blood_glucose_level` | Captures how elevated blood glucose carries greater diagnostic weight in older age groups |

---

### Binary Flags & Composite Risk Features

Converts clinical thresholds and domain knowledge into **binary signals (0/1)**,
making categorical risk boundaries explicitly available to the model.

| Feature | Condition | Clinical Rationale |
|---|---|---|
| `high_hba1c_flag` | `HbA1c >= 6.5` | Flags patients at or above the WHO diagnostic threshold for diabetes |
| `senior_flag` | `age >= 60` | Flags patients in the age group with the highest prevalence of type 2 diabetes |
| `cardio_risk_flag` | `hypertension == 1` OR `heart_disease == 1` | Flags patients with any cardiovascular comorbidity, which is strongly associated with insulin resistance and diabetes risk |

In [19]:
def add_engineered_features(df):
    # arithimetic features     
    df["glucose_hba1c_interaction"] = df["blood_glucose_level"] * df["HbA1c_level"]
    df["age_hba1c_interaction"] = df["age"] * df["HbA1c_level"]
    df["age_bmi_interaction"] = df["age"] * df["bmi"]
    df["bmi_hba1c_interaction"] = df["bmi"] * df["HbA1c_level"]
    df["age_glucose_interaction"] = df["age"] * df["blood_glucose_level"]
    
    # Boolean and Logical Combinations
    df["high_hba1c_flag"] = (df["HbA1c_level"] >= 6.5).astype(int)
    df["senior_flag"] = (df["age"] >= 60).astype(int)
    df["cardio_risk_flag"] = ((df["hypertension"] == 1) | (df["heart_disease"] == 1)).astype(int)

    return df

## Data Normalization

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        ("smoking_ohe", OneHotEncoder(handle_unknown="ignore"), ["smoking_history"]),
        ("age_minmax", MinMaxScaler(), ["age"]),
        ("robust_features", RobustScaler(), ["bmi",
                                              "HbA1c_level",
                                              "blood_glucose_level",
                                              "glucose_hba1c_interaction",
                                              "age_hba1c_interaction",
                                              "age_bmi_interaction",
                                              "bmi_hba1c_interaction",
                                              "age_glucose_interaction"])
    ],
    remainder="passthrough", # to keep the rest of the columns unchanged
    verbose_feature_names_out=False  # to prevent adding the transformer name as a prefix to the feature names
    # smoking_history_No Info insted of smoking_ohe__smoking_history_No Info
)
joblib.dump(preprocessor, "preprocessor.pkl")

['preprocessor.pkl']

## Transform Training data 

In [21]:
x_train_featured = add_engineered_features(x_train)
x_train_processed = preprocessor.fit_transform(x_train_featured) # returns numpy array not Pandas DataFrame so we need to convert it back to DataFrame
joblib.dump(preprocessor, "preprocessor.pkl")
feature_names = preprocessor.get_feature_names_out()
x_train_processed = pd.DataFrame(
    x_train_processed,
    columns=feature_names,
    index=x_train.index
)
x_train_processed.head()

,smoking_history_No Info,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,...,age_hba1c_interaction,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag
0,0.0,0.0,1.0,0.0,0.0,0.0,0.361862,-0.795349,0.142857,-0.847458,...,-0.239693,-0.500569,-0.251066,-0.512287,0,0,0,0,0,0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.261762,0.0,0.0,-0.847458,...,-0.489933,-0.563572,0.157075,-0.648393,0,0,0,0,0,0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.724725,-0.682171,0.142857,-0.932203,...,0.594439,0.118586,-0.180452,-0.073724,0,0,0,0,0,0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.712212,0.0,0.857143,0.0,...,0.838926,0.324306,0.685615,0.502836,1,1,0,1,0,1
4,1.0,0.0,0.0,0.0,0.0,0.0,0.436937,0.0,-1.642857,-0.237288,...,-0.486577,-0.218286,-0.855959,-0.172023,0,0,0,0,0,0


## Transform Validation data 

In [22]:
x_val = add_engineered_features(x_val)
preprocessor = joblib.load("preprocessor.pkl")
x_val_processed = preprocessor.transform(x_val)
x_val_processed = pd.DataFrame(
    x_val_processed,
    columns=feature_names,   # same feature names from train
    index=x_val.index
)

x_val_processed.head()

,smoking_history_No Info,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,...,age_hba1c_interaction,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag
0,0.0,0.0,0.0,0.0,1.0,0.0,0.612112,0.051163,0.285714,0.0,...,0.38255,0.141597,0.366241,0.291115,1,0,0,0,0,0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.774775,0.787597,0.0,-0.677966,...,0.650048,0.731954,0.632091,0.166352,0,0,0,0,1,0
2,0.0,0.0,0.0,0.0,1.0,0.0,0.336837,-0.186047,0.571429,0.322034,...,-0.219559,-0.444842,0.38175,-0.19414,0,0,0,1,0,0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.762262,1.989147,2.142857,0.254237,...,1.499521,1.129482,3.29865,0.781664,1,0,0,1,1,0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.787287,-0.544186,1.714286,2.033898,...,1.402685,0.272659,0.750135,2.090737,1,0,1,1,1,1


## Transform Cleaned Dataset For EDA stage

In [23]:
cleaned = pd.read_csv("../../data/cleaned/cleaned_1.csv")
cleaned_and_binned = discretize(cleaned)
cleaned_binned_engineered = add_engineered_features(cleaned_and_binned)

cleaned_binned_engineered.to_csv("../../data/EDA/transformed_for_eda.csv", index=False)

In [24]:
train_processed_df = x_train_processed.copy()
train_processed_df["diabetes"] = y_train.values

val_processed_df = x_val_processed.copy()
val_processed_df["diabetes"] = y_val.values

train_processed_df.to_csv("../../data/preprocessed/train_preprocessed.csv", index=False)
val_processed_df.to_csv("../../data/preprocessed/validation_preprocessed.csv", index=False)
